In [2]:
import pandas as pd
import os

In [3]:
# Constant map of questions to categories 
ATTENTION_CHECK = 6
REVERSE_CODE = [4, 10, 14, 17]
COGNITIVE = [2, 3, 4, 5, 7]
AFFECTIVE = [8, 9, 10, 11, 12]
EMOTIONAL = [13, 14, 15, 16, 17]
RESONANCE_COLS = ['Positive Resonance_1', 'Positive Resonance_2', 'Positive Resonance_3']

In [4]:
# Get the path prefix - for Jupyter notebooks, use os.getcwd() or os.path.dirname(os.getcwd())
start_idx = 15
file_prefix = os.path.join(os.path.dirname(os.getcwd()), 'data')
is_reverse_coded = False
filename = 'pilotb_data'
df = pd.read_csv(f'{file_prefix}/{filename}.csv')
df = df.iloc[start_idx:]
df = df
df.head()

,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,...,Positive Resonance_3,Other_Source_AI_1,Other_Source_Human_1,Age,Race,Race_6_TEXT,Gender,Gender_4_TEXT,Response,Condition
15,2025-11-22 13:57:33,2025-11-22 14:01:07,0,137.26.198.242,100,213,1,2025-11-22 14:01:08,R_5n6oVgr5gco0NMC,NaN,...,71,10,NaN,2001,1,NaN,2,NaN,It sounds incredibly difficult to be feeling s...,Human
16,2025-11-22 14:06:48,2025-11-22 14:16:42,0,99.21.127.72,100,593,1,2025-11-22 14:16:42,R_1dL87JcWvchpH8o,NaN,...,80,7,NaN,1995,2,NaN,1,NaN,It sounds like you endured an incredibly diffi...,Human
17,2025-11-22 14:45:54,2025-11-22 14:51:57,0,136.52.71.199,100,363,1,2025-11-22 14:51:57,R_34dwCHnt4aNoKfo,NaN,...,65,NaN,8,1988,1,NaN,1,NaN,It's completely understandable that the turbul...,AI
18,2025-11-23 21:36:47,2025-11-23 21:49:22,0,76.151.128.182,100,754,1,2025-11-23 21:49:22,R_5BumWxLOgTjTvuV,NaN,...,37,NaN,6,1996,1,NaN,1,NaN,That must have been an absolutely terrifying m...,AI


In [6]:
# Check for duplicate columns and merge emotional_experience columns
def merge_cols_across_conditions(df, column_name):
    # Check if we have both columns
    if column_name in df.columns and f"{column_name}.1" in df.columns:
        df[f'{column_name}'] = df[column_name].fillna(df[f"{column_name}.1"])
        
        # Drop the original duplicate columns
        df = df.drop([f'{column_name}.1'], axis=1)
    return df

def get_column_names(num_range):
    return [f'Empathy_{i}' for i in num_range]

def reverse_code(df, column_name):
    df[column_name] = df[column_name].apply(lambda x: 10 - x)
    return df

columns = ["Emotional_Experience"]
for i in range(1, 18):
    columns.append(f'Empathy_{i}')

for col in columns:
    df = merge_cols_across_conditions(df, col)

# Convert strings to int
for col_name in range(1, 18):
    df[f'Empathy_{col_name}'] = df[f'Empathy_{col_name}'].astype(int)
df[RESONANCE_COLS] = df[RESONANCE_COLS].astype(int)
filtered_df = df
# Filter for Attention Checks 
is_correct = []
for val in df[f'Empathy_{ATTENTION_CHECK}']:
    if val == 10: is_correct.append(1)
    else: is_correct.append(0)
filtered_df['is_correct'] = is_correct

# Reverse Code 
if not is_reverse_coded:
    for col_name in REVERSE_CODE:
        filtered_df = reverse_code(filtered_df, f'Empathy_{col_name}')
    is_reverse_coded = True


# Group by Disaggregated Emotional Category 
cognitive_cols = get_column_names(COGNITIVE)
filtered_df['cognitive'] = filtered_df[cognitive_cols].mean(axis=1)

affective_cols = get_column_names(AFFECTIVE)
filtered_df['affective'] = filtered_df[affective_cols].mean(axis=1)

emotional_cols = get_column_names(EMOTIONAL)
filtered_df['motivational'] = filtered_df[emotional_cols].mean(axis=1)

# Calculate Mean Over Categories 
filtered_df['general_empathy'] = filtered_df[['cognitive', 'affective', 'motivational']].mean(axis=1)

# Calculate Resonance
filtered_df['positive_resonance'] = filtered_df[RESONANCE_COLS].mean(axis=1)
filtered_df['other_source'] = filtered_df['Other_Source_AI_1'].fillna(filtered_df['Other_Source_Human_1'])

filtered_df.to_csv(f'{file_prefix}/{filename}_filtered.csv', index=False)